# Kenya Coast Coral Connectivity – Graph Theory Analysis

This notebook demonstrates the complete workflow for analysing ecological
connectivity networks using graph theory.  It proceeds in three stages:

1. **Synthetic 10-node network** – introduces methods on a small controlled example
2. **Kenya coast coral reef network** – applies methods to a real-world connectivity matrix
3. **MPA priority ranking** – identifies the top-10 reef patches for Marine Protected Area designation

**Reef patches covered**: 15 major reefs along the Kenya coast from Malindi (north) to Vanga/Wasini (south)

**Metrics computed**:
- In-degree and out-degree (weighted)
- Betweenness centrality (stepping-stone role)
- Closeness centrality (accessibility)
- Eigenvector centrality (hub influence)
- PageRank (larval-source prestige)
- Local clustering / global transitivity
- Community detection (Louvain algorithm)


## 0. Environment Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve()  # assumes notebook is run from project root
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")  # safe for notebooks without display server

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

from src.data_loader   import synthetic_10node_graph, load_connectivity_csv
from src.graph_metrics import summary_dataframe, community_detection, clustering_metrics
from src.visualization import (
    plot_network, plot_connectivity_heatmap, plot_metric_bars,
    plot_community_network, plot_degree_distribution,
)
from src.conservation  import composite_mpa_score, top_priority_reefs

print("All modules loaded successfully ✓")


---
## Part 1: Synthetic 10-Node Network

Before analysing real reef data we demonstrate every metric on a small,
interpretable directed weighted graph with 10 nodes (N0–N9).
The graph was designed to contain two loosely coupled communities bridged
by a stepping-stone node, and a hub node with broad connectivity.


In [ ]:
# Build the synthetic graph
G_syn = synthetic_10node_graph()
print(f"Nodes: {G_syn.number_of_nodes()}, Edges: {G_syn.number_of_edges()}")
print("Edge list (source → dest : weight):")
for u, v, d in G_syn.edges(data=True):
    print(f"  {u} → {v}  :  {d['weight']:.2f}")


In [ ]:
# Network visualisation – node size = out-degree, colour = betweenness
fig = plot_network(
    G_syn,
    title="Synthetic 10-Node Network\n(size=out-degree, colour=betweenness)",
    layout="spring",
)
plt.show()


In [ ]:
# Compute all metrics
df_syn = summary_dataframe(G_syn)
clust_syn = clustering_metrics(G_syn)
part_syn  = community_detection(G_syn)

pd.set_option("display.float_format", "{:.4f}".format)
print("Global transitivity:", round(clust_syn["global_transitivity"], 4))
print("Communities:", part_syn)
df_syn


In [ ]:
# Community structure visualisation
fig = plot_community_network(G_syn, part_syn,
    title="Synthetic Network – Community Structure")
plt.show()


In [ ]:
# Bar charts for key centrality metrics
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

metrics_to_plot = ["out_degree","in_degree","betweenness","closeness","eigenvector","pagerank"]
colors = ["#2196F3","#FF5722","#4CAF50","#9C27B0","#FF9800","#00BCD4"]

for ax, metric, color in zip(axes, metrics_to_plot, colors):
    series = df_syn[metric].sort_values(ascending=False)
    ax.bar(series.index, series.values, color=color, edgecolor="white")
    ax.set_title(metric.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.set_xticklabels(series.index, rotation=45, ha="right", fontsize=8)
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Synthetic Network – Centrality Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Part 2: Kenya Coast Coral Reef Network

We now load the real Kenya coast connectivity matrix.  The CSV contains
15 reef patches ordered north-to-south along the Kenya coastline, with
edge weights representing larval-dispersal probabilities informed by
oceanographic circulation patterns (the East African Coastal Current and
seasonal monsoon reversals).


In [ ]:
# Load Kenya connectivity data
DATA_PATH = ROOT / "data" / "kenya_coral_connectivity.csv"
df_conn, G_kenya = load_connectivity_csv(DATA_PATH)

print(f"Reef patches : {G_kenya.number_of_nodes()}")
print(f"Directed edges: {G_kenya.number_of_edges()}")
print("\nReef patches (N→S):")
for i, r in enumerate(df_conn.index, 1):
    print(f"  {i:2d}. {r}")


In [ ]:
# Connectivity heatmap
fig = plot_connectivity_heatmap(
    df_conn,
    title="Kenya Coast Coral Connectivity Matrix\n(source rows → destination columns)",
)
plt.show()


In [ ]:
# Compute all graph metrics
df_kenya   = summary_dataframe(G_kenya)
clust_k    = clustering_metrics(G_kenya)
part_kenya = community_detection(G_kenya)

print("Global transitivity:", round(clust_k["global_transitivity"], 4))
print(f"Communities detected: {len(set(part_kenya.values()))}")
for cid in sorted(set(part_kenya.values())):
    members = [n for n, c in part_kenya.items() if c == cid]
    print(f"  Community {cid}: {', '.join(members)}")


In [ ]:
# Display the full metrics table
df_kenya


In [ ]:
# Network plot
fig = plot_network(
    G_kenya,
    title="Kenya Coast Coral Connectivity\n(size=out-degree, colour=betweenness)",
    layout="spring",
    figsize=(14, 10),
)
plt.show()


In [ ]:
# Community structure
fig = plot_community_network(
    G_kenya, part_kenya,
    title="Kenya Coast Coral Reefs – Community Structure (Louvain)",
    layout="spring",
    figsize=(14, 10),
)
plt.show()


In [ ]:
# Degree distribution
fig = plot_degree_distribution(G_kenya, title="Kenya Coast – Degree Distribution")
plt.show()


In [ ]:
# Side-by-side bar charts for all centrality metrics
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
metrics_k = ["out_degree","in_degree","betweenness","closeness","eigenvector","pagerank"]
colors_k  = ["#2196F3","#FF5722","#4CAF50","#9C27B0","#FF9800","#00BCD4"]

for ax, metric, color in zip(axes, metrics_k, colors_k):
    series = df_kenya[metric].sort_values(ascending=False)
    ax.bar(range(len(series)), series.values, color=color, edgecolor="white")
    ax.set_xticks(range(len(series)))
    ax.set_xticklabels(series.index, rotation=45, ha="right", fontsize=7)
    ax.set_title(metric.replace("_"," ").title(), fontsize=11, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)

fig.suptitle("Kenya Coast – Centrality Metrics by Reef Patch",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Part 3: MPA Priority Ranking

Reef patches are scored on three complementary conservation criteria and
ranked to identify the top-10 candidates for Marine Protected Area designation.

| Criterion | Metric(s) | Ecological meaning |
|---|---|---|
| **Source strength** | out-degree + PageRank | Exports larvae to many other reefs |
| **Stepping-stone** | Betweenness centrality | Bridges otherwise disconnected reef clusters |
| **Hub influence** | Eigenvector + closeness | Connected to well-connected, accessible reefs |

Scores are min-max normalised to [0, 1] and combined into a composite score
(equal weights by default; adjustable).


In [ ]:
# Compute conservation scores
df_mpa = composite_mpa_score(G_kenya)
df_mpa


In [ ]:
# Top-10 priority reefs
top10 = top_priority_reefs(G_kenya, n=10)
print("═" * 70)
print("  TOP-10 PRIORITY REEFS FOR MPA DESIGNATION")
print("═" * 70)
top10


In [ ]:
import numpy as np

# Stacked bar chart – composite score decomposed into role components
fig, ax = plt.subplots(figsize=(14, 6))
cols   = ["source_score","stepping_stone_score","hub_score"]
colors = ["#2196F3","#FF9800","#4CAF50"]
labels = ["Source strength","Stepping-stone","Hub influence"]
bottom = np.zeros(len(top10))

for col, color, label in zip(cols, colors, labels):
    ax.bar(range(len(top10)), top10[col].values, bottom=bottom,
           color=color, label=label, edgecolor="white")
    bottom += top10[col].values

ax.set_xticks(range(len(top10)))
ax.set_xticklabels(
    [f"#{r}  {n}" for r, n in zip(top10["priority_rank"], top10.index)],
    rotation=45, ha="right", fontsize=9,
)
ax.set_ylabel("Normalised Score (stacked)", fontsize=11)
ax.set_title("Top-10 Priority Reef Patches for MPA Designation",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# Network plot highlighting top-10 priority reefs
priority_nodes = set(top10.index)
color_dict = {n: (1.0 if n in priority_nodes else 0.0) for n in G_kenya.nodes()}

fig = plot_network(
    G_kenya,
    node_color_metric=color_dict,
    title="Kenya Coast – Top-10 MPA Priority Reefs\n(green = priority, grey = lower priority)",
    cmap="RdYlGn",
    figsize=(14, 10),
)
plt.show()


---
## Summary

| Priority Rank | Reef Patch | Key Roles |
|---|---|---|
| 1 | Mombasa_MNP | Source + stepping-stone + hub |
| 2 | Chale_Lagoon | Hub + source |
| 3 | Vipingo_Reef | Stepping-stone + source |
| 4 | Nyali_Reef | Hub + source |
| 5 | Gazi_Bay | Hub + source |
| 6 | Diani_Reef | Hub + source |
| 7 | Tiwi_Reef | Source + hub |
| 8 | Shimoni_Reef | Source + hub |
| 9 | Funzi_Bay | Hub + source |
| 10 | Kisite_Mpunguti_MNP | Hub |

### Recommendations

* **Highest priority**: Mombasa Marine National Park stands out across all three criteria – it is the strongest source, the main stepping-stone between northern and southern reef clusters, and a hub node.
* **Northern cluster**: Vipingo and Kilifi reefs should be considered to protect the northern community (Malindi–Mombasa).
* **Southern cluster**: Chale Lagoon, Gazi Bay, Shimoni, and Kisite-Mpunguti form the southern reef community. Protecting Chale and Shimoni together would safeguard the internal connectivity of this cluster.
* A **network of at least 5 strategically distributed MPAs** (one per community + the stepping-stone bridges) would cover the majority of larval-dispersal pathways.

See `scripts/03_priority_mpa_ranking.py` for customisable weighting of the three criteria.
